In [2]:
import pyspark
from pyspark.sql import SparkSession
import sys
import pandas as pd

In [2]:
pyspark.__file__

'/Users/osiosman/DEZoomcamp2026/.venv/lib/python3.13/site-packages/pyspark/__init__.py'

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 11:43:49 WARN Utils: Your hostname, Osis-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.188 instead (on interface en0)
26/03/03 11:43:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 11:43:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-02 19:22:11--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-03T01%3A20%3A54Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-03T00%3A20%3A19Z&ske=2026-03-03T01%3A20%3A54Z&sks=b&skv=2018-11-09&sig=KJHasLZKFaewKfAMRTDQ3n6vO9%2Bp8otajxA%2FVkBrlnM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjUwMDkzMiwibmJmIjoxNzcyNDk3MzMyLCJwYXRo

In [6]:
!gzip -dc fhvhv_tripdata_2021-01.csv.gz

--2026-03-02 17:45:58--  https://s3.amazonaws.com/nyc-tlc/misc/taxi+_zone_lookup.csv
Resolving s3.amazonaws.com (s3.amazonaws.com)... 54.231.227.32, 16.15.183.230, 54.231.131.232, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|54.231.227.32|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2026-03-02 17:45:58 ERROR 403: Forbidden.



In [5]:
!wc -l fhvhv_tripdata_2021-01.csv.gz

  508066 fhvhv_tripdata_2021-01.csv.gz


In [6]:
!gzip -dc fhvhv_tripdata_2021-01.csv.gz | wc -l       # count rows

 11908469


In [7]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv.gz')

In [8]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [9]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [10]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

In [11]:
!gzip -dc fhvhv_tripdata_2021-01.csv.gz | head -1001 > head.csv

gzip: error writing to output: Broken pipe
gzip: fhvhv_tripdata_2021-01.csv.gz: uncompress failed


In [12]:
!wc -l head.csv

    1001 head.csv


In [13]:
df_pandas = pd.read_csv('head.csv')

In [14]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [15]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [20]:
from pyspark.sql import types

In [18]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)
])

In [19]:
df = spark.read \
    .option("header", "true") \
    .schema(schema)\
    .csv('fhvhv_tripdata_2021-01.csv.gz')

In [21]:
df.head(10)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [27]:
print(df.count())

[Stage 7:>                                                          (0 + 1) / 1]

11908468


In [29]:
df = df.repartition(24)

In [30]:
df.write.parquet('fhvhv/2021/01/')

26/03/02 19:52:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/02 19:52:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/02 19:52:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/02 19:52:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/02 19:52:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/02 19:52:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/02 19:52:38 WARN MemoryManager: Total allocation exceeds 95.

In [5]:
df = spark.read.parquet('fhvhv/2021/01/')

In [6]:
df

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

In [7]:
df.show()

[Stage 1:>                                                          (0 + 1) / 1]

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-30 04:13:09|2021-01-30 04:26:02|         261|          50|   NULL|
|           HV0003|              B02871|2021-01-14 08:25:44|2021-01-14 08:35:06|           4|          88|   NULL|
|           HV0003|              B02883|2021-01-30 23:31:39|2021-01-30 23:45:53|          78|          81|   NULL|
|           HV0005|              B02510|2021-01-16 17:51:42|2021-01-16 18:05:49|         213|          58|   NULL|
|           HV0003|              B02882|2021-01-23 15:33:08|2021-01-23 15:53:39|         188|         188|   NULL|
|           HV0005|              B02510|2021-01-04 15:08:11|2021-01-04 15:18:02|

In [8]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [9]:
df.select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [12]:
df.select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID')\
.filter(df.hvfhs_license_num == 'HV0003')\
.show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-14 08:25:44|2021-01-14 08:35:06|           4|          88|
|2021-01-30 23:31:39|2021-01-30 23:45:53|          78|          81|
|2021-01-23 15:33:08|2021-01-23 15:53:39|         188|         188|
|2021-01-24 14:26:54|2021-01-24 14:37:41|         228|         195|
|2021-01-12 07:49:18|2021-01-12 08:11:06|         216|         196|
|2021-01-08 11:54:39|2021-01-08 12:09:38|          26|          26|
|2021-01-22 21:05:18|2021-01-22 21:30:05|         159|         265|
|2021-01-12 14:56:25|2021-01-12 15:14:23|          50|         145|
|2021-01-29 11:21:53|2021-01-29 11:31:07|         236|         162|
|2021-01-31 08:13:26|2021-01-31 08:16:01|         168|         147|
|2021-01-04 18:05:26|2021-01-04 18:26:59|          85|          39|
|2021-01-16 10:07:19|2021-01-16 10:11:07|       

In [14]:
from pyspark.sql import functions as F

In [ ]:
F.to_date()

In [15]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|           HV0005|              B02510|2021-01-30 04:13:09|2021-01-30 04:26:02|         261|          50|   NULL| 2021-01-30|  2021-01-30|
|           HV0003|              B02871|2021-01-14 08:25:44|2021-01-14 08:35:06|           4|          88|   NULL| 2021-01-14|  2021-01-14|
|           HV0003|              B02883|2021-01-30 23:31:39|2021-01-30 23:45:53|          78|          81|   NULL| 2021-01-30|  2021-01-30|
|           HV0005|              B02510|2021-01-16 17:51:42|2021-01-16 18:05:49|         213|          58|   NULL| 2021-01-16|  2021-01-16|
|           HV0003| 

In [16]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date','dropoff_date','PULocationID','DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-30|  2021-01-30|         261|          50|
| 2021-01-14|  2021-01-14|           4|          88|
| 2021-01-30|  2021-01-30|          78|          81|
| 2021-01-16|  2021-01-16|         213|          58|
| 2021-01-23|  2021-01-23|         188|         188|
| 2021-01-04|  2021-01-04|          17|          17|
| 2021-01-24|  2021-01-24|         228|         195|
| 2021-01-12|  2021-01-12|         216|         196|
| 2021-01-08|  2021-01-08|          26|          26|
| 2021-01-02|  2021-01-02|          28|         130|
| 2021-01-22|  2021-01-22|         159|         265|
| 2021-01-12|  2021-01-12|          50|         145|
| 2021-01-28|  2021-01-28|         140|         141|
| 2021-01-29|  2021-01-29|         236|         162|
| 2021-01-31|  2021-01-31|         168|         147|
| 2021-01-14|  2021-01-14|          97|       

In [17]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [18]:
crazy_stuff('B02884')

's/b44'

In [21]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [22]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id','pickup_date','dropoff_date','PULocationID','DOLocationID') \
    .show()

[Stage 5:>                                                          (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-30|  2021-01-30|         261|          50|
|  a/b37| 2021-01-14|  2021-01-14|           4|          88|
|  a/b43| 2021-01-30|  2021-01-30|          78|          81|
|  e/9ce| 2021-01-16|  2021-01-16|         213|          58|
|  e/b42| 2021-01-23|  2021-01-23|         188|         188|
|  e/9ce| 2021-01-04|  2021-01-04|          17|          17|
|  a/b37| 2021-01-24|  2021-01-24|         228|         195|
|  a/b37| 2021-01-12|  2021-01-12|         216|         196|
|  a/b49| 2021-01-08|  2021-01-08|          26|          26|
|  e/9ce| 2021-01-02|  2021-01-02|          28|         130|
|  e/acc| 2021-01-22|  2021-01-22|         159|         265|
|  e/acc| 2021-01-12|  2021-01-12|          50|         145|
|  e/9ce| 2021-01-28|  2021-01-28|         140|         141|
|  e/acc| 2021-01-29|  2